In [1]:
import json
from pathlib import Path

path = Path("dataset/teacher_subset_100sat_100unsat_matched_sft_noleak.jsonl")

rows = []
with path.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

In [7]:
# !uv pip install pandas

In [16]:
import pandas as pd

df = pd.DataFrame(rows)
df.head().iloc[0]["messages"][0]

{'role': 'system',
 'content': 'You are an expert in propositional logic and Boolean satisfiability (SAT).\nYou are solving SATBench-style natural-language logic puzzles.\n\nImportant reasoning rules:\n- Use only the constraints stated in the conditions and formal CNF information.\n- The scenario is background only and adds no hidden constraints.\n- Treat all variables as independent Boolean decisions unless the conditions explicitly state otherwise.\n- Do not add commonsense assumptions such as mutual exclusivity, exactly-one constraints, or real-world causal links unless they are stated in the conditions.\n- Variables not mentioned in the conditions are irrelevant to satisfiability and may be assigned arbitrarily.\n\nYour task:\n1. Decide whether the puzzle is SAT or UNSAT.\n2. If SAT, give one satisfying assignment or enough assignment information to verify the clauses.\n3. If UNSAT, identify a contradiction or an UNSAT core/relevant conflicting clauses.\n4. Explain the reasoning br

In [18]:
import json
import random
from pathlib import Path
from collections import Counter

# =========================
# Configuration
# =========================

INPUT_PATH = Path("dataset/teacher_subset_100sat_100unsat_matched_sft_noleak.jsonl")

OUT_DIR = Path("dataset/sft_by_label")

# For 100 SAT + 100 UNSAT, this gives:
# SAT:   80 train, 20 test
# UNSAT: 80 train, 20 test
TEST_PER_LABEL = 20

RANDOM_SEED = 42


# =========================
# Helper functions
# =========================

def load_jsonl(path):
    """Load a JSONL file into a list of Python dictionaries."""
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_no}: {e}") from e
    return rows


def write_jsonl(rows, path):
    """Write a list of dictionaries to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def get_label(row):
    """
    Get SAT/UNSAT label from one row.

    The cleaned SFT file should have:
        row["ground_truth_label"] == "SAT" or "UNSAT"

    This function also includes fallbacks in case your file structure changes.
    """
    label = row.get("ground_truth_label")

    if label is None and isinstance(row.get("ground_truth"), dict):
        label = row["ground_truth"].get("label")

    if label is None and isinstance(row.get("ground_truth"), dict):
        satisfiable = row["ground_truth"].get("satisfiable")
        if satisfiable is True:
            label = "SAT"
        elif satisfiable is False:
            label = "UNSAT"

    if label is None:
        raise ValueError(f"Could not find label for row with keys: {list(row.keys())}")

    label = str(label).strip().upper()
    if label not in {"SAT", "UNSAT"}:
        raise ValueError(f"Unexpected label: {label}")

    return label


# =========================
# Load and split
# =========================

rows = load_jsonl(INPUT_PATH)
print(f"Loaded {len(rows)} rows from {INPUT_PATH}")

# Group by SAT / UNSAT
by_label = {"SAT": [], "UNSAT": []}

for row in rows:
    label = get_label(row)
    by_label[label].append(row)

print("Original counts:")
print({label: len(items) for label, items in by_label.items()})

# Basic validation
if len(by_label["SAT"]) == 0 or len(by_label["UNSAT"]) == 0:
    raise ValueError("Both SAT and UNSAT groups must be non-empty.")

if TEST_PER_LABEL >= len(by_label["SAT"]):
    raise ValueError("TEST_PER_LABEL is too large for SAT samples.")

if TEST_PER_LABEL >= len(by_label["UNSAT"]):
    raise ValueError("TEST_PER_LABEL is too large for UNSAT samples.")


# Shuffle each label separately and split
rng = random.Random(RANDOM_SEED)

splits = {}

for label in ["SAT", "UNSAT"]:
    label_rows = list(by_label[label])
    rng.shuffle(label_rows)

    test_rows = label_rows[:TEST_PER_LABEL]
    train_rows = label_rows[TEST_PER_LABEL:]

    splits[label] = {
        "train": train_rows,
        "test": test_rows,
    }


# =========================
# Save files
# =========================

write_jsonl(splits["SAT"]["train"], OUT_DIR / "sat" / "train.jsonl")
write_jsonl(splits["SAT"]["test"], OUT_DIR / "sat" / "test.jsonl")

write_jsonl(splits["UNSAT"]["train"], OUT_DIR / "unsat" / "train.jsonl")
write_jsonl(splits["UNSAT"]["test"], OUT_DIR / "unsat" / "test.jsonl")


# Save a small split report
report = {
    "input_path": str(INPUT_PATH),
    "output_dir": str(OUT_DIR),
    "random_seed": RANDOM_SEED,
    "test_per_label": TEST_PER_LABEL,
    "counts": {
        "sat": {
            "train": len(splits["SAT"]["train"]),
            "test": len(splits["SAT"]["test"]),
            "total": len(by_label["SAT"]),
        },
        "unsat": {
            "train": len(splits["UNSAT"]["train"]),
            "test": len(splits["UNSAT"]["test"]),
            "total": len(by_label["UNSAT"]),
        },
        "all": {
            "train": len(splits["SAT"]["train"]) + len(splits["UNSAT"]["train"]),
            "test": len(splits["SAT"]["test"]) + len(splits["UNSAT"]["test"]),
            "total": len(rows),
        },
    },
}

report_path = OUT_DIR / "split_report.json"
report_path.parent.mkdir(parents=True, exist_ok=True)

with report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)


# =========================
# Print summary
# =========================

print("\nSaved files:")
print(OUT_DIR / "sat" / "train.jsonl")
print(OUT_DIR / "sat" / "test.jsonl")
print(OUT_DIR / "unsat" / "train.jsonl")
print(OUT_DIR / "unsat" / "test.jsonl")
print(report_path)

print("\nSplit report:")
print(json.dumps(report, indent=2))

Loaded 200 rows from dataset/teacher_subset_100sat_100unsat_matched_sft_noleak.jsonl
Original counts:
{'SAT': 100, 'UNSAT': 100}

Saved files:
dataset/sft_by_label/sat/train.jsonl
dataset/sft_by_label/sat/test.jsonl
dataset/sft_by_label/unsat/train.jsonl
dataset/sft_by_label/unsat/test.jsonl
dataset/sft_by_label/split_report.json

Split report:
{
  "input_path": "dataset/teacher_subset_100sat_100unsat_matched_sft_noleak.jsonl",
  "output_dir": "dataset/sft_by_label",
  "random_seed": 42,
  "test_per_label": 20,
  "counts": {
    "sat": {
      "train": 80,
      "test": 20,
      "total": 100
    },
    "unsat": {
      "train": 80,
      "test": 20,
      "total": 100
    },
    "all": {
      "train": 160,
      "test": 40,
      "total": 200
    }
  }
}


In [2]:
import json
import re
import time
import html
from pathlib import Path
from threading import Thread
from collections import Counter

import torch
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display, HTML

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextIteratorStreamer,
)


# ============================================================
# 1. Configuration
# ============================================================

MODEL_ID = "Qwen/Qwen3.5-0.8B-Base"

SAT_TEST_PATH = Path("dataset/sft_by_label/sat/test.jsonl")
UNSAT_TEST_PATH = Path("dataset/sft_by_label/unsat/test.jsonl")

OUTPUT_DIR = Path("results/baseline_qwen35_08b_base_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSONL = OUTPUT_DIR / "qwen35_08b_base_test_predictions.jsonl"
OUTPUT_CSV = OUTPUT_DIR / "qwen35_08b_base_test_predictions.csv"

# Set this to True if the model is already downloaded and you are running offline on Adroit.
LOCAL_FILES_ONLY = False

# Generation settings.
# For a base model, deterministic decoding is usually better for evaluation.
MAX_NEW_TOKENS = 8192
DO_SAMPLE = False
TEMPERATURE = 0.0
TOP_P = 1.0

# Streaming display settings.
# Set to False if you only want tqdm progress and no streamed text.
STREAM_OUTPUT = True

# If you only want to smoke test a few examples first, set this to an integer.
# Example: LIMIT_PER_SPLIT = 2
LIMIT_PER_SPLIT = None


# ============================================================
# 2. JSONL loading helpers
# ============================================================

def load_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON in {path} line {line_no}: {e}") from e
    return rows


def write_jsonl(rows, path):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


sat_rows = load_jsonl(SAT_TEST_PATH)
unsat_rows = load_jsonl(UNSAT_TEST_PATH)

if LIMIT_PER_SPLIT is not None:
    sat_rows = sat_rows[:LIMIT_PER_SPLIT]
    unsat_rows = unsat_rows[:LIMIT_PER_SPLIT]

# Add explicit split labels for easier reporting.
for row in sat_rows:
    row["_eval_split"] = "sat_test"
    row["_expected_label_from_file"] = "SAT"

for row in unsat_rows:
    row["_eval_split"] = "unsat_test"
    row["_expected_label_from_file"] = "UNSAT"

test_rows = sat_rows + unsat_rows

print(f"Loaded SAT test rows:   {len(sat_rows)}")
print(f"Loaded UNSAT test rows: {len(unsat_rows)}")
print(f"Loaded total rows:      {len(test_rows)}")


# ============================================================
# 3. Label and prompt helpers
# ============================================================

def get_ground_truth_label(row):
    """
    Prefer explicit ground_truth_label if present.
    Fall back to ground_truth.label, ground_truth.satisfiable,
    or the file label we attached above.
    """
    label = row.get("ground_truth_label")

    if label is None and isinstance(row.get("ground_truth"), dict):
        label = row["ground_truth"].get("label")

    if label is None and isinstance(row.get("ground_truth"), dict):
        satisfiable = row["ground_truth"].get("satisfiable")
        if satisfiable is True:
            label = "SAT"
        elif satisfiable is False:
            label = "UNSAT"

    if label is None:
        label = row.get("_expected_label_from_file")

    if label is None:
        raise ValueError(f"Cannot find ground-truth label. Row keys: {list(row.keys())}")

    label = str(label).strip().upper()
    if label not in {"SAT", "UNSAT"}:
        raise ValueError(f"Unexpected ground-truth label: {label}")

    return label


def get_prompt_messages(row):
    """
    SFT rows usually look like:
      {
        "messages": [
          {"role": "system", "content": "..."},
          {"role": "user", "content": "..."},
          {"role": "assistant", "content": "...teacher answer..."}
        ]
      }

    For evaluation, we must remove the assistant target.
    """
    messages = row.get("messages")
    if not isinstance(messages, list):
        raise ValueError(f"Row does not contain a valid messages list. Keys: {list(row.keys())}")

    prompt_messages = []
    for msg in messages:
        role = msg.get("role")
        content = msg.get("content", "")
        if role == "assistant":
            continue
        prompt_messages.append({"role": role, "content": content})

    if not prompt_messages:
        raise ValueError("No non-assistant messages found.")

    return prompt_messages


def messages_to_prompt(tokenizer, messages):
    """
    Use the tokenizer chat template if available.
    For a Base model, the tokenizer may not have a useful chat template,
    so we fall back to a simple plain-text format.
    """
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        parts = []
        for msg in messages:
            role = msg.get("role", "user").upper()
            content = msg.get("content", "")
            parts.append(f"{role}:\n{content}")
        parts.append("ASSISTANT:\n")
        return "\n\n".join(parts)


def extract_predicted_label(text):
    """
    Extract final SAT/UNSAT prediction.

    First looks for bracketed labels like [SAT] or [UNSAT].
    If not found, falls back to the last occurrence of the words SAT/UNSAT.
    UNSAT is checked carefully so it is not confused with SAT.
    """
    if not text:
        return None

    bracket_matches = re.findall(r"\[\s*(UNSAT|SAT)\s*\]", text, flags=re.IGNORECASE)
    if bracket_matches:
        return bracket_matches[-1].upper()

    word_matches = re.findall(r"\b(UNSAT|SAT)\b", text, flags=re.IGNORECASE)
    if word_matches:
        return word_matches[-1].upper()

    return None


# ============================================================
# 4. Load tokenizer and model
# ============================================================

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    local_files_only=LOCAL_FILES_ONLY,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model: {MODEL_ID}")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
    local_files_only=LOCAL_FILES_ONLY,
)

model.eval()

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ============================================================
# 5. Streaming generation helper
# ============================================================

@torch.no_grad()
def generate_one(row, row_id):
    """
    Generate one model response with optional streaming display.
    Returns a result dictionary.
    """
    gold = get_ground_truth_label(row)
    messages = get_prompt_messages(row)
    prompt_text = messages_to_prompt(tokenizer, messages)

    inputs = tokenizer(prompt_text, return_tensors="pt")

    # Put input tensors on the same device as the embedding layer.
    # This works with device_map="auto" too.
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    if DO_SAMPLE:
        generation_kwargs.update(
            temperature=TEMPERATURE,
            top_p=TOP_P,
        )

    start_time = time.time()

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    generated_text = ""

    if STREAM_OUTPUT:
        header = (
            f"<b>Example {row_id}</b> | "
            f"split={row.get('_eval_split')} | "
            f"gold={gold}"
        )
        display_handle = display(
            HTML(header + "<pre style='white-space: pre-wrap; font-size: 13px;'></pre>"),
            display_id=True,
        )

    for new_text in streamer:
        generated_text += new_text

        if STREAM_OUTPUT:
            safe_text = html.escape(generated_text)
            display_handle.update(
                HTML(
                    header
                    + "<pre style='white-space: pre-wrap; font-size: 13px;'>"
                    + safe_text
                    + "</pre>"
                )
            )

    thread.join()
    elapsed = time.time() - start_time

    pred = extract_predicted_label(generated_text)
    correct = pred == gold

    input_tokens = int(inputs["input_ids"].shape[-1])
    output_tokens = len(tokenizer.encode(generated_text, add_special_tokens=False))

    result = {
        "row_id": row_id,
        "eval_split": row.get("_eval_split"),
        "gold_label": gold,
        "predicted_label": pred,
        "correct": correct,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "seconds": elapsed,
        "tokens_per_second": output_tokens / elapsed if elapsed > 0 else None,
        "prompt_text": prompt_text,
        "model_response": generated_text,
    }

    return result


# ============================================================
# 6. Run evaluation with tqdm progress
# ============================================================

results = []

for i, row in enumerate(tqdm(test_rows, desc="Evaluating Qwen3.5-0.8B-Base")):
    try:
        result = generate_one(row, row_id=i)
    except Exception as e:
        result = {
            "row_id": i,
            "eval_split": row.get("_eval_split"),
            "gold_label": get_ground_truth_label(row),
            "predicted_label": None,
            "correct": False,
            "error": repr(e),
            "model_response": "",
        }
        print(f"Error on row {i}: {repr(e)}")

    results.append(result)

    # Save after every example so you do not lose progress if the notebook stops.
    write_jsonl(results, OUTPUT_JSONL)


print(f"Saved raw predictions to: {OUTPUT_JSONL}")


# ============================================================
# 7. Summarize results
# ============================================================

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)

def accuracy_for(sub_df):
    if len(sub_df) == 0:
        return None
    return float(sub_df["correct"].mean())

summary = {
    "model_id": MODEL_ID,
    "num_total": len(df),
    "num_sat_test": int((df["eval_split"] == "sat_test").sum()),
    "num_unsat_test": int((df["eval_split"] == "unsat_test").sum()),
    "overall_accuracy": accuracy_for(df),
    "sat_test_accuracy": accuracy_for(df[df["eval_split"] == "sat_test"]),
    "unsat_test_accuracy": accuracy_for(df[df["eval_split"] == "unsat_test"]),
    "predicted_label_counts": dict(Counter(df["predicted_label"].fillna("NONE"))),
    "gold_label_counts": dict(Counter(df["gold_label"].fillna("NONE"))),
    "avg_output_tokens": float(df["output_tokens"].mean()) if "output_tokens" in df else None,
    "avg_tokens_per_second": float(df["tokens_per_second"].dropna().mean()) if "tokens_per_second" in df else None,
}

summary_path = OUTPUT_DIR / "qwen35_08b_base_test_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nSummary:")
print(json.dumps(summary, indent=2))

print(f"\nSaved CSV to: {OUTPUT_CSV}")
print(f"Saved summary to: {summary_path}")

display(df[[
    "row_id",
    "eval_split",
    "gold_label",
    "predicted_label",
    "correct",
    "output_tokens",
    "seconds",
    "tokens_per_second",
]].head())

Loaded SAT test rows:   20
Loaded UNSAT test rows: 20
Loaded total rows:      40
Loading tokenizer: Qwen/Qwen3.5-0.8B-Base


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading model: Qwen/Qwen3.5-0.8B-Base


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

CUDA available: True
GPU: NVIDIA A100 80GB PCIe


Evaluating Qwen3.5-0.8B-Base:   0%|          | 0/40 [00:00<?, ?it/s]

KeyboardInterrupt: 